# 线性求解器部署实验

## 10.1 实验概览

按顺序运行下面的代码，需要修改的参数已在代码中标出。


In [ ]:
%%bash
set -e
echo "CANN 目录：$ASCEND_HOME_PATH"
cmake --version
npu-smi info


<a id="s-10-02_xyce_application_and_adapter"></a>


## 10.2 Xyce 应用角色与 Adapter

### 本节目标

- 理解 wrapper 如何映射 Xyce linear solver 概念
- 追踪 `solve(A,b,x)` 到 Ascend-GMRES backend


### 实际调用链

`XyceApplicationWrapper` 用 `b=A*x_true` 构造已知解的线性问题。`XyceSparseMatrixAdapter` 保存 CSR，`XyceLinearSolverAdapter` 选择求解器。

- CPU single/OpenMP：调用附带的 `backend/gmres/Ascend-GMRES`。
- AscendDevice：调用第 09 章的 `dis_gmres::distributed_gmres`，由 Ascend C RTC kernel 完成主要计算。

因此请保留完整课程目录。这个 wrapper 用于研究求解器替换，不读取电路 netlist。


### 比较条件

三种 solver 使用同一矩阵、右端项、restart=30、最大迭代数 10000 和残差容差 `1e-6`。
同时检查相对已知解的误差（`solution_error_vs_expected`）；OpenMP 和 Device 还与 CPU 解比较。

练习：在源码中标出 assembly、prepare、solve 和误差检查的位置，说明为什么 wrapper 的总时间大于线性求解时间。

[参考答案](answer/10.02_answer.md)


<a id="s-10-03_dependency_and_build_deployment"></a>


## 10.3 依赖准备与构建部署

### 本节目标

- 找到 CPU 与设备求解器使用的源码
- 构建线性求解器测试程序
- 区分求解器测试与完整 Xyce 构建


### 依赖位置

CPU 求解器使用随仓库提供的 `backend/gmres/Ascend-GMRES`；设备求解器引用第 09 章的 GMRES。保留完整课程目录后，按下面的命令即可构建，无需下载完整 Xyce。

`ASCEND_XYCE_FETCH_XYCE` 和 `ASCEND_XYCE_BUILD_XYCE` 用于选做的上游 Xyce 构建，默认关闭。


### 构建本章工程

下面用 CMake 编译适配器、求解器和测试程序。CMake 会使用仓库内的 `backend/gmres/Ascend-GMRES` 源码，以及第 09 章的 NPU GMRES 实现。


In [ ]:
%%bash
set -e
cd src/ascend_xyce
cmake -S . -B build -DCMAKE_BUILD_TYPE=Release -DASCEND_XYCE_HOST_ONLY=OFF
cmake --build build -j


### 课后实践

找出 CPU 和设备求解器分别使用哪个目录的源码，解释为什么要保留完整课程目录，以及随仓库提供依赖有什么好处。

参考答案见 `answer/10.03_answer.md`。


<a id="s-10-04_benchmark_launch_and_configuration"></a>


## 10.4 Benchmark 启动与配置

### 本节目标

- 使用脚本启动测试程序
- 理解矩阵输入回退顺序
- 管理 warmup、repeat 和结果路径


### 运行参数

直接运行编译得到的 `build/bin/xyce_benchmark`。`--matrix` 选择输入，`--csv` 指定结果文件；路径相对于 `src/ascend_xyce`。

`warmup=N` 会额外执行并丢弃 N 次求解；`repeat` 指定统计次数。每次求解重新准备适配器和设备求解状态，矩阵只在重复前读取或生成一次。本实验使用 `warmup=0`。

程序先找本工程的矩阵缓存，再找相邻 Ascend-GMRES 的矩阵；都没有时用固定种子生成。因此第一次运行可能会生成矩阵缓存。


### 快速运行一个输入

下面只运行 U1，使用 0 次预热和 1 次计时，先检查构建、启动和结果保存。正式比较时可增大 `--repeat`，并保持各轮的输入和迭代参数一致。


In [ ]:
%%bash
set -eo pipefail
cd src/ascend_xyce
mkdir -p results
# 学生修改：--matrix 选择输入矩阵。
./build/bin/xyce_benchmark --warmup 0 --repeat 1 --matrix U1 --csv results/course_u1.csv 2>&1 | tee results/run.log
head -4 results/course_u1.csv


### 检查结果

三个 solver 均应 `converged=1`，真实相对残差不超过 `1e-6`；相对已知解的误差不超过 `1e-3`。残差检查方程满足得怎样，解误差检查与已知解相差多少，两者含义不同。

CSV 对当前矩阵有三行结果。本工程在同机比较求解器，无需集群作业系统。

### 课后实践

为 U2 写出启动命令，说明输入、输出和两种误差的判据。参考答案见 `answer/10.04_answer.md`。


<a id="s-10-05_logs_results_and_troubleshooting"></a>


## 10.5 结果、日志与故障定位

### 本节目标

- 读取本次 CSV 的应用与求解器字段
- 区分 assembly、linear solver 和 total
- 按部署阶段定位故障


### 读取本次结果

打开上一步生成的 CSV，先检查三种求解器的收敛状态、残差和解误差。

`total_simulation_ms` 包含矩阵装配、适配器准备和求解；`linear_solver_ms` 是其中的线性求解时间。比较求解器看后者，分析应用整体开销看前者。SpMV 又是求解的一部分，这些时间存在包含关系。

旧版 CPU 原型数据保存在源码 README 中，作为选读参考。

### 怎样理解本次 Device 比 CPU 慢

U1 这次单次测试中，CPU single 的线性求解为 27.09 ms，Device 为 3246.01 ms，约慢 120 倍。日志中的 `speedup=0.00745x` 用的是应用总时间 `30.94/4152.38`，对应约慢 134 倍；它与线性求解耗时的比值不同。

在 `xyce_adapter.cpp` 中，`Other` 等于线性求解总时间减去已计入的 SpMV、Dot、AXPY、Norm、Givens、Residual。Device 的 2371 ms 是未单列的工作合计，并非实测出的“RTC 编译时间”。`npu_gmres.cpp` 在计时开始后调用 `NpuCompute::prepare()`，其中包含 RTC 编译、二进制加载、矩阵上传；向量分配、Host 控制等也会进入未单列部分。Sync、Launch、H2D/D2H 是补充观察项，与这些阶段存在包含关系，不能再相加。

CPU 与 Device 分别用 13 步和 32 步：两条路径使用不同的浮点运算与规约实现，求解参数相同并不保证舍入过程或迭代数相同。两者都通过真实残差和解误差检查，日志不足以把迭代差异归因于某一个运算。更多迭代和大量小 kernel 的启动、同步，以及每次重建 RTC/Device 状态，一起影响了本次结果。它说明当前实现和计时范围下没有获得加速，不能直接推广为 NPU 对所有线性问题都更慢。


设备行的 `Residual=0 ms` 表示适配器没有单独填充残差计算的耗时字段。最终真实残差仍由 Device GMRES 返回，查看 `final_residual` 判断；时间字段为零不表示没有做检查。


### 遇到错误时

编译错误查看缺失的工具、头文件或库；运行错误查看设备初始化和求解信息；结果未写出时检查 CSV 路径。日志保存在 `results/run.log`。


### 课后实践

用未知参数触发一次错误，说明报错原因以及应该修改哪一步。参考答案见 `answer/10.05_answer.md`。
